In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tokenizers import Tokenizer

from src.model import Transformer
from src.loss import build_loss_fn
from src.scheduler import warmup_lr_lambda
from src.data_pipeline import Multi30kDataset, make_collate_fn

# config (SMALL TEST VALUES)
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 100
DROPOUT = 0.1
BATCH_SIZE = 32
WARMUP_STEPS = 4000
NUM_EPOCHS = 1          # just 1 epoch for the smoke test
CLIP_NORM = 1.0
TRAIN_SUBSET_SIZE = 200 # tiny slice of training data to test that the loop runs

# tokenizer + pad id
tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")
PAD_ID = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()
collate_fn = make_collate_fn(PAD_ID)

# datasets
train_dataset_full = Multi30kDataset("data/raw/train.en", "data/raw/train.de", tokenizer)
val_dataset = Multi30kDataset("data/raw/val.en", "data/raw/val.de", tokenizer)

train_dataset = Subset(train_dataset_full, range(TRAIN_SUBSET_SIZE))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# model, loss, optimizer, scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT, pad_id=PAD_ID).to(device)
loss_fn = build_loss_fn(PAD_ID, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: warmup_lr_lambda(step, D_MODEL, WARMUP_STEPS)
)

step = 0

def train_epoch():
    global step
    model.train()
    total_loss = 0
    for src, tgt_input, tgt_label in train_loader:
        src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)

        optimizer.zero_grad()
        output = model(src, tgt_input)
        loss = loss_fn(output.reshape(-1, VOCAB_SIZE), tgt_label.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        scheduler.step()
        step += 1

        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt_input, tgt_label in val_loader:
            src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)
            output = model(src, tgt_input)
            loss = loss_fn(output.reshape(-1, VOCAB_SIZE), tgt_label.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(val_loader)

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch()
    val_loss = validate()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

Epoch 1/1 | train loss: 9.0850 | val loss: 9.0695


In [3]:
# inference (greedy decoding)

@torch.no_grad()
def greedy_decode(model, src, src_mask, bos_id, eos_id, max_len, device):
    model.eval()
    enc_output = model.encoder(src, src_mask)

    tgt = torch.tensor([[bos_id]], device=device)

    for _ in range(max_len):
        tgt_mask = model.make_tgt_mask(tgt)
        output = model.decoder(tgt, enc_output, src_mask, tgt_mask)

        next_token_logits = output[:, -1, :]
        next_token = next_token_logits.argmax(dim=-1).unsqueeze(1)

        tgt = torch.cat([tgt, next_token], dim=1)

        if next_token.item() == eos_id:
            break

    return tgt

In [4]:
# verifying

def translate_sentence(model, sentence, tokenizer, device, max_len=100):
    model.eval()
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")

    src_ids = tokenizer.encode(sentence).ids
    src = torch.tensor([src_ids], device=device)
    src_mask = model.make_src_mask(src)

    tgt_ids = greedy_decode(model, src, src_mask, bos_id, eos_id, max_len, device)
    output_tokens = tokenizer.decode(tgt_ids[0].tolist())
    return output_tokens

print(translate_sentence(model, "A man is riding a bicycle.", tokenizer, device))

athon Saxofon te leep houses schreit Baby main theater shorts unus Regen Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem grocer durch Saxophon Rennen raum drin ative carry grünem


In [5]:
# finding the timing for a single epoch

import time

train_dataset = train_dataset_full

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

start = time.time()
train_loss = train_epoch()
elapsed = time.time() - start

val_loss = validate()

print(f"1 epoch took {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print(f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

1 epoch took 866.2 seconds (14.44 minutes)
train loss: 6.0700 | val loss: 4.9961


In [7]:
# final training loop

import os
import json
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tokenizers import Tokenizer

from src.model import Transformer
from src.loss import build_loss_fn
from src.scheduler import warmup_lr_lambda
from src.data_pipeline import Multi30kDataset, make_collate_fn
from src.evaluate import translate_sentence

# config
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 100
DROPOUT = 0.1
BATCH_SIZE = 32
WARMUP_STEPS = 4000
NUM_EPOCHS = 20
CLIP_NORM = 1.0

SAMPLE_SENTENCES = [
    "A man is riding a bicycle.",
    "Two young, White males are outside near many bushes.",
    "A woman is cooking in the kitchen.",
]

os.makedirs("checkpoints", exist_ok=True)

# tokenizer + pad id
tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")
PAD_ID = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()
collate_fn = make_collate_fn(PAD_ID)

# datasets
train_dataset = Multi30kDataset("data/raw/train.en", "data/raw/train.de", tokenizer)
val_dataset = Multi30kDataset("data/raw/val.en", "data/raw/val.de", tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# model, loss, optimizer, scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT, pad_id=PAD_ID).to(device)
loss_fn = build_loss_fn(PAD_ID, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: warmup_lr_lambda(step, D_MODEL, WARMUP_STEPS)
)

step = 0

def train_epoch():
    global step
    model.train()
    total_loss = 0
    for src, tgt_input, tgt_label in train_loader:
        src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)

        optimizer.zero_grad()
        output = model(src, tgt_input)
        loss = loss_fn(output.reshape(-1, VOCAB_SIZE), tgt_label.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        scheduler.step()
        step += 1

        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt_input, tgt_label in val_loader:
            src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)
            output = model(src, tgt_input)
            loss = loss_fn(output.reshape(-1, VOCAB_SIZE), tgt_label.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(val_loader)

# training loop
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss = train_epoch()
    val_loss = validate()
    elapsed = time.time() - start

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    with open("checkpoints/history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | {elapsed:.1f}s")

    # checkpoint at every epoch
    torch.save(model.state_dict(), f"checkpoints/epoch_{epoch}.pt")

    # example translations
    with open("checkpoints/translations_log.txt", "a", encoding="utf-8") as f:
        f.write(f"Epoch {epoch}\n")
        for sentence in SAMPLE_SENTENCES:
            translation = translate_sentence(model, sentence, tokenizer, device, max_len=MAX_LEN)
            f.write(f"  EN: {sentence}\n  DE: {translation}\n")
            print(f"  EN: {sentence}\n  DE: {translation}")
        f.write("\n")

print("Training complete.")

Epoch 1/20 | train loss: 6.1088 | val loss: 5.0040 | 520.4s
  EN: A man is riding a bicycle.
  DE: Ein Mann fährt auf einem Fahrrad .
  EN: Two young, White males are outside near many bushes.
  DE: Zwei junge junge junge junge junge junge in einem kleinen Mädchen , während ein Foto .
  EN: A woman is cooking in the kitchen.
  DE: Eine Frau in einem blauen Oberteil spielt .
Epoch 2/20 | train loss: 4.5892 | val loss: 4.2854 | 576.6s
  EN: A man is riding a bicycle.
  DE: Ein Mann fährt Fahrrad auf einem Fahrrad .
  EN: Two young, White males are outside near many bushes.
  DE: Zwei junge Erwachsene sind im Freien in der Nähe von Kindern .
  EN: A woman is cooking in the kitchen.
  DE: Eine Frau in der Küche .
Epoch 3/20 | train loss: 4.0065 | val loss: 3.8382 | 593.5s
  EN: A man is riding a bicycle.
  DE: Ein Mann fährt Fahrrad auf einem Fahrrad .
  EN: Two young, White males are outside near many bushes.
  DE: Zwei junge Männer , die draußen neben einem kleinen Kindern .
  EN: A woma